In [1]:
import sys; sys.path.append("..")
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
feat_files = sorted(Path("../data/processed/features").glob("*.parquet"))
features = pd.concat([pd.read_parquet(p) for p in feat_files]).sort_index()
news = pd.read_parquet("../data/processed/news.parquet")
reddit = pd.read_parquet("../data/processed/reddit.parquet")

news["date"] = pd.to_datetime(news["date"])
reddit["date"] = pd.to_datetime(reddit["date"])

print(f"FEATURES: {features.index.min().date()} -> {features.index.max().date()}  ({len(features):,} rows)")
print(f"NEWS:     {news['date'].min().date()} -> {news['date'].max().date()}  ({len(news):,} rows)")
print(f"REDDIT:   {reddit['date'].min().date()} -> {reddit['date'].max().date()}  ({len(reddit):,} rows)")

FEATURES: 2021-09-29 -> 2026-07-13  (12,000 rows)
NEWS:     2011-03-03 -> 2020-06-11  (14,303 rows)
REDDIT:   2021-01-28 -> 2021-08-16  (4,072 rows)


In [3]:
overlap_start = max(features.index.min(), news["date"].min())
overlap_end   = min(features.index.max(), news["date"].max())
print(f"PRICE ∩ NEWS overlap: {overlap_start.date()} -> {overlap_end.date()}")
print(f"Days in overlap: {(overlap_end - overlap_start).days}")

in_overlap = features[(features.index >= overlap_start) & (features.index <= overlap_end)]
print(f"Feature rows inside overlap: {len(in_overlap):,}")
print(f"Tickers in overlap: {in_overlap['ticker'].nunique()}")

PRICE ∩ NEWS overlap: 2021-09-29 -> 2020-06-11
Days in overlap: -475
Feature rows inside overlap: 0
Tickers in overlap: 0


In [4]:
y = features["target_1d"].astype(int)
print(f"Up-days: {y.mean():.3f}  ->  always-up baseline = {max(y.mean(), 1-y.mean()):.3f}")
print(f"\nFeature columns: {[c for c in features.columns if c not in ('ticker','target_1d','target_5d')]}")
print(f"\nAny NaNs? {features.isna().sum().sum()}")

Up-days: 0.525  ->  always-up baseline = 0.525

Feature columns: ['ret_1d', 'ret_2d', 'ret_3d', 'ret_5d', 'ret_10d', 'price_vs_ma20', 'ma20_vs_ma50', 'vol_20d', 'rsi_14', 'macd', 'volume_z']

Any NaNs? 0


In [5]:
import yfinance as yf
from src.utils.config import TICKERS

HIST_START, HIST_END = "2011-01-01", "2020-06-30"   # covers the news window + warm-up

frames = []
for ticker in TICKERS:
    df = yf.download(ticker, start=HIST_START, end=HIST_END, auto_adjust=True, progress=False)
    if df.empty:
        print(f"  no data: {ticker}"); continue
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.reset_index()
    df.columns = [c.lower() for c in df.columns]
    df["ticker"] = ticker
    df = df[["date", "ticker", "open", "high", "low", "close", "volume"]]
    df = df.dropna(subset=["open", "high", "low", "close"])
    frames.append(df)
    print(f"  {ticker}: {len(df)} rows")

hist_prices = pd.concat(frames, ignore_index=True)
print("TOTAL:", hist_prices.shape)


  AAPL: 2388 rows
  MSFT: 2388 rows
  GOOGL: 2388 rows
  AMZN: 2388 rows
  NVDA: 2388 rows
  TSLA: 2388 rows
  META: 2041 rows
  JPM: 2388 rows
  JNJ: 2388 rows
  XOM: 2388 rows
TOTAL: (23533, 7)


In [6]:
from src.preprocessing.features import compute_features

HIST_FEATURES_DIR = Path("../data/processed/features_hist")
HIST_FEATURES_DIR.mkdir(parents=True, exist_ok=True)

hist_frames = []
for ticker in TICKERS:
    tdf = hist_prices[hist_prices["ticker"] == ticker]
    feats = compute_features(tdf)
    feats.to_parquet(HIST_FEATURES_DIR / f"{ticker}.parquet")
    hist_frames.append(feats)
    print(f"  {ticker}: {len(feats)} feature rows")

hist_features = pd.concat(hist_frames).sort_index()
print("HISTORICAL FEATURES:", hist_features.shape)
print(f"Range: {hist_features.index.min().date()} -> {hist_features.index.max().date()}")

  AAPL: 2334 feature rows
  MSFT: 2334 feature rows
  GOOGL: 2334 feature rows
  AMZN: 2334 feature rows
  NVDA: 2334 feature rows
  TSLA: 2334 feature rows
  META: 1987 feature rows
  JPM: 2334 feature rows
  JNJ: 2334 feature rows
  XOM: 2334 feature rows
HISTORICAL FEATURES: (22993, 14)
Range: 2011-03-15 -> 2020-06-22


In [7]:
o_start = max(hist_features.index.min(), news["date"].min())
o_end   = min(hist_features.index.max(), news["date"].max())
print(f"NEW OVERLAP: {o_start.date()} -> {o_end.date()}  ({(o_end-o_start).days} days)")

ov = hist_features[(hist_features.index >= o_start) & (hist_features.index <= o_end)]
print(f"Feature rows in overlap: {len(ov):,}  |  tickers: {ov['ticker'].nunique()}")
print(f"Baseline in this window: {max(ov['target_1d'].mean(), 1-ov['target_1d'].mean()):.3f}")

NEW OVERLAP: 2011-03-15 -> 2020-06-11  (3376 days)
Feature rows in overlap: 22,923  |  tickers: 10
Baseline in this window: 0.522


In [8]:
# numeric score: +1 positive, -1 negative, 0 neutral, weighted by model confidence
news["sent_value"] = news["sentiment"].map({"positive": 1, "negative": -1, "neutral": 0}) * news["sentiment_score"]

daily = (news.groupby(["date", "ticker"])
              .agg(sent_mean=("sent_value", "mean"),
                   news_count=("sent_value", "size"))
              .reset_index())

# LAG BY ONE DAY — the leakage guard
daily = daily.sort_values("date")
daily["date"] = daily["date"] + pd.Timedelta(days=1)

print(daily.shape)
print(daily.head())
print("\nRows per ticker:\n", daily["ticker"].value_counts())

(4402, 4)
        date ticker  sent_mean  news_count
0 2011-03-04   NVDA  -0.825096           1
1 2011-03-08   NVDA   0.000000           2
2 2011-03-09   NVDA   0.086861           4
3 2011-03-10   NVDA   0.000000           3
4 2011-03-11   NVDA   0.000000           2

Rows per ticker:
 ticker
JNJ      1371
NVDA     1197
JPM       561
GOOGL     479
XOM       305
TSLA      284
META       87
AAPL       81
AMZN       37
Name: count, dtype: int64


In [9]:
hf = hist_features.reset_index().rename(columns={"index": "date"})
hf["date"] = pd.to_datetime(hf["date"])

merged = hf.merge(daily, on=["date", "ticker"], how="left")

print(f"Rows: {len(merged):,}")
print(f"Rows WITH sentiment: {merged['sent_mean'].notna().sum():,} "
      f"({merged['sent_mean'].notna().mean():.1%})")
print("\nCoverage by ticker:")
print(merged.groupby("ticker")["sent_mean"].apply(lambda s: f"{s.notna().mean():.1%}"))

Rows: 22,993
Rows WITH sentiment: 3,405 (14.8%)

Coverage by ticker:
ticker
AAPL      2.5%
AMZN      1.2%
GOOGL    16.2%
JNJ      46.5%
JPM      17.2%
META      3.3%
MSFT      0.0%
NVDA     40.7%
TSLA      8.8%
XOM      10.1%
Name: sent_mean, dtype: str


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)

PRICE_FEATURES = ["ret_1d","ret_2d","ret_3d","ret_5d","ret_10d",
                  "price_vs_ma20","ma20_vs_ma50","vol_20d","rsi_14","macd","volume_z"]
SENT_FEATURES = ["sent_mean", "news_count"]

exp = merged.dropna(subset=["sent_mean"]).sort_values("date").reset_index(drop=True)
print(f"Experiment rows: {len(exp):,} | {exp['date'].min().date()} -> {exp['date'].max().date()}")

# chronological 80/20 split — derived from the data, not hard-coded
split_idx = int(len(exp) * 0.8)
split_date = exp.loc[split_idx, "date"]
train, test = exp[exp["date"] < split_date], exp[exp["date"] >= split_date]
print(f"Train: {len(train):,} (to {train['date'].max().date()}) | Test: {len(test):,} (from {test['date'].min().date()})")

y_train = train["target_1d"].astype(int)
y_test  = test["target_1d"].astype(int)
baseline = max(y_test.mean(), 1 - y_test.mean())
print(f"Baseline on test: {baseline:.3f}")

def run(name, feats):
    model = Pipeline([("scaler", StandardScaler()),
                      ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
    model.fit(train[feats], y_train)
    pred  = model.predict(test[feats])
    proba = model.predict_proba(test[feats])[:, 1]
    return model, proba, {
        "accuracy":  accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall":    recall_score(y_test, pred, zero_division=0),
        "f1":        f1_score(y_test, pred, zero_division=0),
        "roc_auc":   roc_auc_score(y_test, proba),
        "pr_auc":    average_precision_score(y_test, proba),
    }

model_a, proba_a, res_a = run("A", PRICE_FEATURES)
model_b, proba_b, res_b = run("B", PRICE_FEATURES + SENT_FEATURES)

results = pd.DataFrame({"A: price only": res_a, "B: price + sentiment": res_b}).T
print("\n", results.round(4))
print(f"\nBaseline accuracy: {baseline:.4f}")
print(f"Accuracy delta (B - A): {res_b['accuracy'] - res_a['accuracy']:+.4f}")
print(f"ROC-AUC delta  (B - A): {res_b['roc_auc'] - res_a['roc_auc']:+.4f}")

Experiment rows: 3,405 | 2011-03-16 -> 2020-06-12
Train: 2,724 (to 2019-12-26) | Test: 681 (from 2019-12-27)
Baseline on test: 0.508

                       accuracy  precision  recall      f1  roc_auc  pr_auc
A: price only           0.5419     0.5457  0.5867  0.5655   0.5709  0.5641
B: price + sentiment    0.5580     0.5581  0.6243  0.5894   0.5778  0.5713

Baseline accuracy: 0.5081
Accuracy delta (B - A): +0.0162
ROC-AUC delta  (B - A): +0.0069


In [11]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.base import clone

tss = TimeSeriesSplit(n_splits=5)
X, y_all = exp, exp["target_1d"].astype(int)
rows = []

for fold, (tr_i, te_i) in enumerate(tss.split(X), 1):
    tr, te = X.iloc[tr_i], X.iloc[te_i]
    ytr, yte = y_all.iloc[tr_i], y_all.iloc[te_i]
    fold_res = {"fold": fold, "test_start": te["date"].min().date(), "n_test": len(te),
                "baseline": max(yte.mean(), 1 - yte.mean())}
    for label, feats in [("A", PRICE_FEATURES), ("B", PRICE_FEATURES + SENT_FEATURES)]:
        m = Pipeline([("s", StandardScaler()),
                      ("c", LogisticRegression(max_iter=1000, class_weight="balanced"))])
        m.fit(tr[feats], ytr)
        fold_res[f"acc_{label}"] = accuracy_score(yte, m.predict(te[feats]))
        fold_res[f"auc_{label}"] = roc_auc_score(yte, m.predict_proba(te[feats])[:, 1])
    rows.append(fold_res)

wf = pd.DataFrame(rows)
wf["delta_acc"] = wf["acc_B"] - wf["acc_A"]
print(wf.round(4).to_string(index=False))
print(f"\nMean delta: {wf['delta_acc'].mean():+.4f} ± {wf['delta_acc'].std():.4f}")
print(f"Folds where B > A: {(wf['delta_acc'] > 0).sum()} / {len(wf)}")

 fold test_start  n_test  baseline  acc_A  auc_A  acc_B  auc_B  delta_acc
    1 2014-05-12     567    0.5485 0.5009 0.4714 0.4868 0.4787    -0.0141
    2 2017-05-03     567    0.5238 0.5379 0.5435 0.5485 0.5492     0.0106
    3 2018-11-15     567    0.5185 0.4956 0.4845 0.4780 0.4870    -0.0176
    4 2019-07-26     567    0.5661 0.5115 0.5164 0.5309 0.5301     0.0194
    5 2020-02-06     567    0.5009 0.5503 0.5772 0.5503 0.5784     0.0000

Mean delta: -0.0004 ± 0.0158
Folds where B > A: 2 / 5
